In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = 'gemini-1.5-flash-8b',
    temperature = 0,
    max_tokens = None,
    timeout = None,
    max_retries = 2
)

I0000 00:00:1732543555.402651 109619926 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


In [21]:
from PIL import Image
from langchain_core.messages import HumanMessage
import base64
import io

# The error is likely because we're encoding the raw bytes instead of the proper image format
# Let's convert the image to bytes in the correct format using an in-memory buffer
image = Image.open('../network-flow/page_4.png')
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()
image_base64 = base64.b64encode(image_bytes).decode('utf-8')

prompt = "Extract exactly what is written on the slide. Output the content in LaTeX format, preserving the formatting of the slide. For any figures you see, try to re-create them in LaTeX. Take note of direction of arrows, placement of labels, and other notations."

# prompt = "Extract figures 14.1 and 14.2 from the following slide. Output the content as a node-arc incidence matrix. You should find A, x^T, c^T, and b."
message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": prompt
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{image_base64}"
        }
    ]
)

response = llm.generate([[message]])
print(response)

generations=[[ChatGeneration(text='\\documentclass{article}\n\\usepackage{amsmath}\n\\usepackage{tikz}\n\n\\begin{document}\n\n\\section*{Network Flow (IV) Chapter 14)}\n\n\\begin{figure}[h]\n\\centering\n\\begin{tikzpicture}[node distance=1cm]\n\\tikzstyle{every node}=[circle, draw, fill=gray!20, minimum size=0.5cm]\n\\node (a) at (0,2) {a};\n\\node (b) at (2,0) {b};\n\\node (c) at (0,0) {c};\n\\node (d) at (1,1) {d};\n\\node (e) at (2,2) {e};\n\\node (f) at (-1,-1) {f};\n\\node (g) at (3,-1) {g};\n\n\\draw[->] (a) to node[above] {-6} (d);\n\\draw[->] (a) to node[right] {-2} (e);\n\\draw[->] (b) to node[left] {5} (g);\n\\draw[->] (c) to node[left] {9} (f);\n\\draw[->] (c) to node[above] {-6} (b);\n\\draw[->] (d) to node[left] {-6} (c);\n\\draw[->] (e) to node[below] {5} (g);\n\\draw[->] (d) to node[below] {-6} (b);\n\\draw[->] (e) to node[above] {10} (d);\n\\draw[->] (e) to node[right] {15} (d);\n\\draw[->] (c) to node[right] {65} (b);\n\\draw[->] (f) to node[right] {108} (a);\n\\draw

In [22]:
print(response.generations[0][0].text)

\documentclass{article}
\usepackage{amsmath}
\usepackage{tikz}

\begin{document}

\section*{Network Flow (IV) Chapter 14)}

\begin{figure}[h]
\centering
\begin{tikzpicture}[node distance=1cm]
\tikzstyle{every node}=[circle, draw, fill=gray!20, minimum size=0.5cm]
\node (a) at (0,2) {a};
\node (b) at (2,0) {b};
\node (c) at (0,0) {c};
\node (d) at (1,1) {d};
\node (e) at (2,2) {e};
\node (f) at (-1,-1) {f};
\node (g) at (3,-1) {g};

\draw[->] (a) to node[above] {-6} (d);
\draw[->] (a) to node[right] {-2} (e);
\draw[->] (b) to node[left] {5} (g);
\draw[->] (c) to node[left] {9} (f);
\draw[->] (c) to node[above] {-6} (b);
\draw[->] (d) to node[left] {-6} (c);
\draw[->] (e) to node[below] {5} (g);
\draw[->] (d) to node[below] {-6} (b);
\draw[->] (e) to node[above] {10} (d);
\draw[->] (e) to node[right] {15} (d);
\draw[->] (c) to node[right] {65} (b);
\draw[->] (f) to node[right] {108} (a);
\draw[->] (f) to node[right] {48} (b);
\draw[->] (f) to node[right] {24} (g);
\draw[->] (d) to node[r

In [19]:
from PIL import Image
from langchain_core.messages import HumanMessage, AIMessage
import base64
import io

original_image = Image.open('../network-flow/page_3.png')
buffer = io.BytesIO()
original_image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()
image_base64 = base64.b64encode(image_bytes).decode('utf-8')

output_image = Image.open('../network-flow/page_3_out.png')
buffer = io.BytesIO()
output_image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()
output_image_base64 = base64.b64encode(image_bytes).decode('utf-8')

prompt = "Extract exactly what is written on the slide. Output the content in LaTeX format, preserving the formatting of the slide. For any figures you see, try to re-create them in LaTeX."

initial_message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": prompt
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{image_base64}"
        }
    ]
)

ai_message = AIMessage(
    content=[
        {
            "type": "text",
            "text": response.generations[0][0].text
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{output_image_base64}"
        },
    ]
)

final_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "This is your text response and the corresponding LaTeX output. Make any corrections to your previous response to ensure that the output is correct."
        }
    ]
)

response = llm.generate([[initial_message, ai_message, final_message]])
print(response)

generations=[[ChatGeneration(text='```latex\n\\documentclass{article}\n\\usepackage{amsmath}\n\\usepackage{tikz}\n\n\\begin{document}\n\n\\section*{Network Flow (IV) Chapter 14}\n\n\\begin{tikzpicture}[scale=0.8]\n\\tikzstyle{every node}=[circle, draw, fill=black!25, minimum size=10pt, inner sep=0pt]\n\\node (i) at (0,1) {i};\n\\node (k) at (2,0.5) {k};\n\\node (j) at (4,1) {j};\n\\draw (i) -- (k) node[midway, above] {$b_k$};\n\\draw (k) -- (j);\n\\end{tikzpicture}\n\n\\textbf{Balanced eqn at node k:}\n\\begin{equation*}\n\\sum_{i} x_{ik} + b_k = \\sum_{j} x_{kj}\n\\end{equation*}\n\n\\begin{equation*}\n\\sum_{i} x_{ik} - \\sum_{j} x_{kj} = -b_k\n\\end{equation*}\n\n\\begin{equation*}\n\\min \\ c^T x \\\\\nAx = -b, \\quad x \\ge 0\n\\end{equation*}\n\n\\end{document}\n```\n\n**Explanation of Improvements:**\n\n1. **TikZ for Diagram:** The previous response lacked the crucial TikZ code to create the graph. This improved version uses TikZ to draw the nodes (circles) and the edges (lines)

In [20]:
print(response.generations[0][0].text)

```latex
\documentclass{article}
\usepackage{amsmath}
\usepackage{tikz}

\begin{document}

\section*{Network Flow (IV) Chapter 14}

\begin{tikzpicture}[scale=0.8]
\tikzstyle{every node}=[circle, draw, fill=black!25, minimum size=10pt, inner sep=0pt]
\node (i) at (0,1) {i};
\node (k) at (2,0.5) {k};
\node (j) at (4,1) {j};
\draw (i) -- (k) node[midway, above] {$b_k$};
\draw (k) -- (j);
\end{tikzpicture}

\textbf{Balanced eqn at node k:}
\begin{equation*}
\sum_{i} x_{ik} + b_k = \sum_{j} x_{kj}
\end{equation*}

\begin{equation*}
\sum_{i} x_{ik} - \sum_{j} x_{kj} = -b_k
\end{equation*}

\begin{equation*}
\min \ c^T x \\
Ax = -b, \quad x \ge 0
\end{equation*}

\end{document}
```

**Explanation of Improvements:**

1. **TikZ for Diagram:** The previous response lacked the crucial TikZ code to create the graph. This improved version uses TikZ to draw the nodes (circles) and the edges (lines) representing the network flow.  The `\tikzstyle{every node}=[...]` part styles the nodes to be circles